# Week 11 — AI use and judgments of competence

**Research task:** Use an LLM to edit a fictional competitor report. Check the edit and use Python to prepare four conditions for a possible human-reader study.

**Python introduced:** one nested loop that crosses two lists of conditions.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjbarrie/GenAI_Soc2026/blob/main/workbook/session11/session11_language_inequality.ipynb)

## Prepare the environment

Run this cell first. Colab uses OpenRouter. Local JupyterLab uses Ollama by default.

In [ ]:
SESSION = "session11"

# Run this cell first. It prepares Colab or checks the local Python environment.
import importlib as setup_importlib
import importlib.util as setup_importlib_util
import os as setup_os
import subprocess as setup_subprocess
import sys as setup_sys
from pathlib import Path as SetupPath

try:
    import google.colab as setup_colab  # type: ignore[import-not-found]
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

course_packages = {
    "openrouter": "openrouter>=0.6,<1",
    "ollama": "ollama>=0.6,<1",
}
if SESSION == "session05":
    course_packages["plotly"] = "plotly>=6.3,<7"
    course_packages["umap"] = "umap-learn>=0.5.9,<1"
    course_packages["pandas"] = "pandas>=2.3,<3"
if SESSION == "session13":
    course_packages["pandas"] = "pandas>=2.2,<3"

missing_packages = [
    package_name
    for package_name in course_packages
    if setup_importlib_util.find_spec(package_name) is None
]

if IN_COLAB:
    if missing_packages:
        packages_to_install = [course_packages[name] for name in missing_packages]
        setup_subprocess.run(
            [
                setup_sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "--disable-pip-version-check",
                *packages_to_install,
            ],
            check=True,
        )
        setup_importlib.invalidate_caches()

    COURSE_ROOT = SetupPath(
        setup_os.getenv("COURSE_COLAB_ROOT", "/content/GenAI_Soc2026")
    )
    if not COURSE_ROOT.exists():
        setup_subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/cjbarrie/GenAI_Soc2026.git",
                str(COURSE_ROOT),
            ],
            check=True,
        )
    setup_os.chdir(COURSE_ROOT / "workbook" / SESSION)
else:
    COURSE_ROOT = SetupPath.cwd()
    while not (COURSE_ROOT / "config" / "course_models.json").exists() and COURSE_ROOT != COURSE_ROOT.parent:
        COURSE_ROOT = COURSE_ROOT.parent
    if missing_packages:
        missing_text = ", ".join(missing_packages)
        raise ModuleNotFoundError(
            f"This notebook is using a Python environment without: {missing_text}.\n\n"
            "Close Jupyter. Open a terminal in the GenAI_Soc2026 repository and run:\n"
            "    uv sync\n"
            "    uv run jupyter lab\n\n"
            "In VS Code, select the Python interpreter inside the repository's .venv folder."
        )
    if not (COURSE_ROOT / "config" / "course_models.json").exists():
        raise FileNotFoundError(
            "The complete GenAI_Soc2026 repository could not be found. A notebook "
            "downloaded by itself is not enough for local work. Download or clone the "
            "repository, open a terminal in that folder, and run: uv run jupyter lab"
        )

course_root_text = str(COURSE_ROOT)
if course_root_text not in setup_sys.path:
    setup_sys.path.insert(0, course_root_text)

still_missing = [
    package_name
    for package_name in course_packages
    if setup_importlib_util.find_spec(package_name) is None
]
if still_missing:
    raise ModuleNotFoundError(
        "Setup did not make these packages available: " + ", ".join(still_missing)
    )

print("Environment:", "Google Colab" if IN_COLAB else "local course environment")
print("Python executable:", setup_sys.executable)
print("Course root:", COURSE_ROOT)
print("Working folder:", SetupPath.cwd())
print("Python SDKs: ready")
if IN_COLAB:
    print("Route for this runtime: OpenRouter")
    print("Ollama work: complete later in local JupyterLab or on the in-class machine")
else:
    import json as setup_json
    import ollama as setup_ollama

    setup_config = setup_json.loads(
        (COURSE_ROOT / "config" / "course_models.json").read_text()
    )
    setup_local_model = setup_config["local"]["model"]
    try:
        setup_models = setup_ollama.list().models
        setup_model_names = [
            getattr(item, "model", None) or getattr(item, "name", None)
            for item in setup_models
        ]
        print("Ollama server: reachable at localhost:11434")
        if setup_local_model in setup_model_names:
            print("Course local model: ready —", setup_local_model)
        else:
            print("Course local model: NOT INSTALLED —", setup_local_model)
            print("NEXT STEP: open a terminal and run: ollama pull " + setup_local_model)
    except Exception as setup_error:
        print("Ollama server: NOT REACHABLE")
        print("NEXT STEP: start the Ollama application, then run: ollama list")
        print("Diagnostic:", str(setup_error).splitlines()[0])


## Load tools and select a route

The script reads the two course model names. Colab selects OpenRouter; a local notebook selects Ollama.

In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

try:
    import ollama
    from openrouter import OpenRouter
except ModuleNotFoundError:
    ollama = None
    OpenRouter = None

ROOT = Path.cwd()
while not (ROOT / "config" / "course_models.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "config" / "course_models.json").exists():
    raise FileNotFoundError(
        "Open a terminal in the complete GenAI_Soc2026 folder, then run this file."
    )

config = json.loads((ROOT / "config" / "course_models.json").read_text())
OPENROUTER_MODEL = config["hosted"]["model"]
OLLAMA_MODEL = config["local"]["model"]
HOSTED_MODEL = OPENROUTER_MODEL
LOCAL_MODEL = OLLAMA_MODEL
USE_EXAMPLE = os.getenv("COURSE_USE_RECORDED", "0") == "1"
ROUTE = "openrouter" if IN_COLAB else "ollama"

## Store one fictional report

The report and editing instruction are strings. The report contains no confidential business information.

In [ ]:
SOURCE_PARAGRAPH = (
    "RivalCo launched a monthly subscription in April at $18. "
    "Our comparable plan costs $20. RivalCo advertises a free trial. "
    "We do not yet have data on customer switching after the launch."
)

EDITING_INSTRUCTION = (
    "Edit this short competitor report for clarity. Preserve every stated fact "
    "and the uncertainty about customer switching. Do not infer a market effect, "
    "invent data, or add a recommendation. Return only the revised report."
)

# This passage is an instructor-written fallback for an outage, not a model result.
EXAMPLE_REVISION = (
    "RivalCo launched an $18 monthly subscription in April. Our comparable "
    "$20 plan costs $2 more. RivalCo advertises a free trial. We do not yet "
    "have data on whether customers switched after the launch."
)

print("Source paragraph:", SOURCE_PARAGRAPH)
print("Python type:", type(SOURCE_PARAGRAPH))
print("Editing instruction:", EDITING_INSTRUCTION)

## Ask the model for one edit

The function shows the complete message, calls the chosen route, prints the raw return and keeps a record of the call.

In [ ]:
def edit_paragraph(source_text, instruction, route):
    """Return the revision and preserve the exact text sent and received."""
    separator = "\n\nOriginal paragraph:\n"
    prompt = instruction + separator + source_text
    messages = [{"role": "user", "content": prompt}]
    print("Messages sent to the model:", messages)

    if USE_EXAMPLE:
        raw_text = EXAMPLE_REVISION
        model_name = "instructor-written outage example"
        used_route = "example"
    elif route == "openrouter":
        if OpenRouter is None:
            raise ModuleNotFoundError("Run 'uv sync --frozen' before using OpenRouter.")
        api_key = os.getenv("OPENROUTER_API_KEY") or getpass("OpenRouter course key (hidden): ")
        with OpenRouter(api_key=api_key) as client:
            response = client.chat.send(
                model=OPENROUTER_MODEL,
                messages=messages,
                temperature=0,
            )
        raw_text = response.choices[0].message.content
        model_name = OPENROUTER_MODEL
        used_route = route
    elif route == "ollama":
        if ollama is None:
            raise ModuleNotFoundError("Run 'uv sync --frozen' before using Ollama.")
        response = ollama.chat(think=False,
            model=OLLAMA_MODEL,
            messages=messages,
            options={"temperature": 0},
        )
        raw_text = response.message.content
        model_name = OLLAMA_MODEL
        used_route = route
    else:
        raise ValueError("route must be 'openrouter' or 'ollama'")

    if not isinstance(raw_text, str) or not raw_text.strip():
        raise ValueError("The model returned no usable text. Inspect the raw response.")
    print("Raw return:", raw_text)
    print("Return type:", type(raw_text))
    return {
        "source": source_text,
        "instruction": instruction,
        "messages": messages,
        "raw_return": raw_text,
        "route": used_route,
        "model": model_name,
    }

editing_record = edit_paragraph(SOURCE_PARAGRAPH, EDITING_INSTRUCTION, ROUTE)
REVISED_PARAGRAPH = editing_record["raw_return"].strip()

## Check the revision

Read the source and revision together. Preserve all five points, including that customer switching is unknown.

In [ ]:
print("ORIGINAL:\n", SOURCE_PARAGRAPH)
print("REVISION:\n", REVISED_PARAGRAPH)
print("Check these facts yourself: April launch; $18 rival plan; $20 comparable plan;")
print("free trial; no customer-switching data yet.")
print("Reject any new claim about switching or market effects.")

## Make the four conditions

The outer loop selects one worker role. For each role, the inner loop selects each AI-use label. The same report goes into all four records.

In [ ]:
ROLE_CUES = [
    "The report author is the senior director of strategy.",
    "The report author is a junior market analyst.",
]
AI_USE_LABELS = [
    "The description gives no information about AI writing assistance.",
    "The author reports using ChatGPT to edit the report for clarity.",
]

conditions = []
for role_cue in ROLE_CUES:
    for ai_use_label in AI_USE_LABELS:
        condition = {
            "condition_id": len(conditions) + 1,
            "role_cue": role_cue,
            "ai_use_label": ai_use_label,
            "report_text": REVISED_PARAGRAPH,
        }
        conditions.append(condition)
        print("Added condition:", condition["condition_id"])
        print("  role cue:", role_cue)
        print("  AI-use label:", ai_use_label)

## Inspect and check the materials

The assertions verify the four-record structure and identical report text. Human readers would still be needed to measure evaluation.

In [ ]:
for condition in conditions:
    print("\nCONDITION", condition["condition_id"])
    print(condition["role_cue"])
    print(condition["ai_use_label"])
    print(condition["report_text"])

assert len(conditions) == 4
assert len({item["condition_id"] for item in conditions}) == 4
assert all(item["report_text"] == REVISED_PARAGRAPH for item in conditions)
print("Four conditions exist, and every condition contains the same report.")
print("These checks do not show how a human reader would judge either worker.")

# ONE CHANGE: make the editing instruction more conservative, rerun, and inspect
# whether the revision changes. Keep the fictional source report fixed.

## Your change and recording

Make the editing instruction more conservative while keeping the source report fixed. Rerun and compare the raw revision. Then trace one complete pass through the outer and inner loops. Explain why the four materials cannot tell us what human readers would think about either worker.